# 1 -  نصب کتابخانه‌ها و اتصال به درایو

In [ ]:
# ==========================================
# 1. اتصال به گوگل درایو
# ==========================================
from google.colab import drive
drive.mount('/content/drive')

# ==========================================
# 2. نصب کتابخانه های مورد نیاز
# ==========================================
print("Installing required packages... (This might take a minute)")
!pip install torch-geometric -q
!pip install transformers -q
!pip install scipy gensim networkx numpy -q
print("Installation Complete!")

Mounted at /content/drive
Installing required packages... (This might take a minute)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 37.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 62.7 MB/s eta 0:00:00
Installation Complete!


# 2 -  بارگذاری داده‌ها

In [ ]:
import os
import json
import torch
import numpy as np
from tqdm.auto import tqdm
from datetime import datetime

# اگر SAMPLE_SIZE = -1 باشد، کل داده ها بارگذاری میشوند
# برای تست های اولیه روی 100 بگذارید، برای اجرای نهایی روی -1 بگذارید
SAMPLE_SIZE = -1

base_path = "/content/drive/MyDrive/datasets/twibot20"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

def safe_load_json(filename, limit):
    file_path = os.path.join(base_path, filename)
    data = []

    # اگر limit برابر -1 باشد، کل فایل خوانده میشود
    if limit == -1:
        print(f"Loading full dataset: {filename}...")
        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        return data

    # در غیر این صورت به صورت خط به خط خوانده میشود تا مصرف رم کنترل شود
    print(f"Loading {limit} samples from {filename}...")
    with open(file_path, "r", encoding="utf-8") as f:
        # بررسی فرمت فایل (آیا لیست است یا خط به خط؟)
        first_char = f.read(1)
        f.seek(0)

        if first_char == '[':
            # فایل به صورت یک لیست بزرگ است
            all_data = json.load(f)
            data = all_data[:limit]
        else:
            # فایل به صورت JSON Lines است
            for i, line in enumerate(f):
                if i >= limit:
                    break
                try:
                    data.append(json.loads(line))
                except:
                    pass
    return data

# بارگذاری داده ها
train_data = safe_load_json("train.json", SAMPLE_SIZE)
# برای داده های dev و test: اگر SAMPLE_SIZE برابر -1 بود، همه را لود کن، در غیر این صورت یک پنجم داده ها را لود کن
dev_limit = -1 if SAMPLE_SIZE == -1 else max(50, SAMPLE_SIZE // 5)
test_limit = -1 if SAMPLE_SIZE == -1 else max(50, SAMPLE_SIZE // 5)

dev_data = safe_load_json("dev.json", dev_limit)
test_data = safe_load_json("test.json", test_limit)

print(f"Train size: {len(train_data)}, Dev size: {len(dev_data)}, Test size: {len(test_data)}")

Device: cuda
Loading full dataset: train.json...
Loading full dataset: dev.json...
Loading full dataset: test.json...
Train size: 8278, Dev size: 2365, Test size: 1183


# 3 - توابع استخراج ویژگی‌ها

In [ ]:
import re
import numpy as np
from datetime import datetime as dt

def clean_text(text):
    if text is None: return ""
    text = str(text).replace("\n", " ").replace("\t", " ")
    return re.sub(r"\s+", " ", text).strip()

def extract_property_features(user):
    profile = user.get("profile") or {}
    created_at = profile.get("created_at") or "Tue Nov 18 10:27:25 +0000 2008"
    date0 = dt.strptime('Tue Sep 1 00:00:00 +0000 2020', '%a %b %d %X %z %Y')
    try:
        date = dt.strptime(created_at, '%a %b %d %X %z %Y')
        active_days = (date0 - date).days
    except:
        active_days = 0

    # استخراج لیست توییت‌ها برای محاسبه ریتوییت
    tweets = user.get("tweet") or []
    if isinstance(tweets, str): tweets = [tweets]
    if not tweets: tweets = []

    # ویژگی جدید مقاله: تعداد ریتوییت‌ها (RT @)
    retweets_count = sum(1 for t in tweets if "RT @" in t or "rt @" in t)

    # 8 ویژگی عددی (5 مورد قبلی + retweets + 2 مورد جدید مقاله ای که بعدا در سلول 4 محاسبه میشود)
    numerical = {
        "followers_count": float(profile.get("followers_count") or 0),
        "friends_count": float(profile.get("friends_count") or 0),
        "favourites_count": float(profile.get("favourites_count") or 0),
        "statuses_count": float(profile.get("statuses_count") or 0),
        "screen_name_length": float(len(profile.get("screen_name") or "")),
        "active_days": float(active_days),
        "retweets": float(retweets_count)
    }

    properties_list = ['protected','geo_enabled','verified','contributors_enabled','is_translator',
                  'is_translation_enabled','profile_background_tile','profile_use_background_image',
                  'has_extended_profile','default_profile','default_profile_image']

    categorical = {}
    for prop in properties_list:
        val = profile.get(prop) or "False"
        categorical[prop] = 1.0 if str(val).strip().lower() == "true" else 0.0

    return list(numerical.values()), list(categorical.values())

def extract_anti_mimicry_features(user):
    tweets = user.get("tweet") or []
    if isinstance(tweets, str): tweets = [tweets]
    if not tweets: tweets = []

    texts = [clean_text(t) for t in tweets]
    all_text = " ".join(texts)
    words = all_text.split()
    tweet_lens = [len(t) for t in texts]

    if len(tweet_lens) > 1:
        diffs = np.abs(np.diff(tweet_lens))
        posting_interval_pattern = float(np.mean(diffs))
        activity_periodicity = float(np.std(diffs))
    else:
        posting_interval_pattern = 0.0
        activity_periodicity = 0.0

    short_tweets = sum(1 for l in tweet_lens if l < 10)
    burstiness = short_tweets / len(tweet_lens) if len(tweet_lens) > 0 else 0.0

    unique_words = set(words)
    vocabulary_size = len(unique_words)
    total_words = len(words)
    unique_word_ratio = vocabulary_size / total_words if total_words > 0 else 0.0
    text_diversity = len(tweets) / len(set(texts)) if len(set(texts)) > 0 else 0.0

    profile = user.get("profile") or {}
    bio = clean_text(profile.get("description"))
    bio_words = set(bio.split())
    semantic_distance = 1.0 - (len(unique_words.intersection(bio_words)) / max(1, len(bio_words)))

    anti_mimicry_dict = {
        "semantic_distance": semantic_distance,
        "text_diversity": text_diversity,
        "vocabulary_size": float(vocabulary_size),
        "unique_word_ratio": unique_word_ratio,
        "posting_interval_pattern": posting_interval_pattern,
        "activity_periodicity": activity_periodicity,
        "burstiness": burstiness
    }
    return list(anti_mimicry_dict.values())

# 4 - استخراج ویژگی‌های  (RoBERTa)

# 5 - ساخت گراف و استخراج Community Features

In [ ]:
import networkx as nx
from scipy.sparse import csr_matrix
from sklearn.decomposition import NMF

print("Building Real Graph...")
node_ids = {}
current_idx = 0

for dataset in [train_data, dev_data, test_data]:
    for user in dataset:
        uid = str(user["ID"]).strip()
        if uid not in node_ids:
            node_ids[uid] = current_idx
            current_idx += 1

G = nx.DiGraph()
G.add_nodes_from(range(current_idx))

# ذخیره نوع یال برای RGCN (0: following, 1: follower)
edge_type_list = []
src_list, dst_list = [], []

for dataset in [train_data, dev_data, test_data]:
    for user in dataset:
        uid = str(user["ID"]).strip()
        u_idx = node_ids[uid]
        neighbors = user.get("neighbor", None)
        if neighbors:
            for n_id in neighbors.get("following", []):
                if str(n_id).strip() in node_ids:
                    src_list.append(u_idx)
                    dst_list.append(node_ids[str(n_id).strip()])
                    edge_type_list.append(0) # following
            for n_id in neighbors.get("follower", []):
                if str(n_id).strip() in node_ids:
                    src_list.append(u_idx)
                    dst_list.append(node_ids[str(n_id).strip()])
                    edge_type_list.append(1) # follower

print(f"Graph built with {G.number_of_nodes()} nodes and {len(src_list)} edges.")

# ==========================================
# پیاده‌سازی DANMF برای استخراج 128 بُعد ساختاری
# ==========================================
print("Extracting Community Features using DANMF approach (Target 128-dim)...")
nodes = list(G.nodes())
A = nx.adjacency_matrix(G, nodelist=nodes).astype(np.float32)
S = A.dot(A)
S.data = np.log(S.data + 1)

# تنظیم ابعاد داینامیک (مطابق مقاله پایه باید 128 باشد)
dim1 = min(128, G.number_of_nodes() - 1) if G.number_of_nodes() > 1 else 1
dim2 = min(128, dim1) if dim1 > 1 else 1

nmf_layer1 = NMF(n_components=dim1, init='nndsvda', random_state=42, max_iter=200)
H1 = nmf_layer1.fit_transform(S)

nmf_layer2 = NMF(n_components=dim2, init='nndsvda', random_state=42, max_iter=200)
community_embeddings = nmf_layer2.fit_transform(H1)

# اگر ابعاد کمتر از 128 شد، با صفر پد میکنیم تا مطابق مقاله 128 بُعدی شود
if dim2 < 128:
    pad_size = 128 - dim2
    padding = np.zeros((community_embeddings.shape[0], pad_size))
    community_embeddings = np.hstack((community_embeddings, padding))

community_embeddings = torch.tensor(community_embeddings, dtype=torch.float)
print("DANMF Community Features Shape:", community_embeddings.shape)

train_indices = [node_ids[str(u["ID"]).strip()] for u in train_data]
dev_indices = [node_ids[str(u["ID"]).strip()] for u in dev_data]
test_indices = [node_ids[str(u["ID"]).strip()] for u in test_data]

comm_train = community_embeddings[train_indices]
comm_dev = community_embeddings[dev_indices]
comm_test = community_embeddings[test_indices]

# اضافه کردن Self-Loop
for i in range(current_idx):
    src_list.append(i)
    dst_list.append(i)
    edge_type_list.append(0) # Self-loop as type 0

edge_index = torch.tensor([src_list, dst_list], dtype=torch.long).to(device)
edge_type = torch.tensor(edge_type_list, dtype=torch.long).to(device)

print("Graph and DANMF features are ready!")

Building Real Graph...
Graph built with 11826 nodes and 16908 edges.
Extracting Community Features using DANMF approach (Target 128-dim)...
DANMF Community Features Shape: torch.Size([11826, 128])
Graph and DANMF features are ready!


# 6 - تعریف مدل‌های پایه و پیشنهادی

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import math
from torch_geometric.nn import RGCNConv, SAGEConv

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def scaled_dot_product_attention(self, Q, K, V):
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        attention_weights = F.softmax(scores, dim=-1)
        attention_weights = self.dropout(attention_weights)
        return torch.matmul(attention_weights, V)

    def forward(self, query, key, value):
        batch_size = query.size(0)
        Q = self.W_q(query).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(key).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(value).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)

        attn_out = self.scaled_dot_product_attention(Q, K, V)
        attn_out = attn_out.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        return self.W_o(attn_out)

class BotDetector(nn.Module):
    def __init__(self, hidden_dim=160, dropout=0.3, num_heads=8, use_anti_mimicry=True, gnn_type="RGCN"):
        super().__init__()
        self.dropout = dropout
        self.use_anti_mimicry = use_anti_mimicry
        self.gnn_type = gnn_type

        # ابعاد لایه‌های خطی (مطابق مقاله پایه)
        d_out = int(hidden_dim / 5) - 10  # 22
        t_out = int(hidden_dim / 5) + 10  # 42
        n_out = int(hidden_dim / 5) - 10  # 22
        c_out = int(hidden_dim / 5) + 20  # 52
        s_out = int(hidden_dim / 5) - 10  # 22
        am_out = int(hidden_dim / 5) - 10 # 22 (فقط برای روش پیشنهادی)

        # لایه‌های استخراج ویژگی پایه
        self.linear_relu_des = nn.Sequential(nn.Linear(768, d_out), nn.LeakyReLU())
        self.linear_relu_tweet = nn.Sequential(nn.Linear(768, t_out), nn.LeakyReLU())
        self.linear_relu_num_prop = nn.Sequential(nn.Linear(7, n_out), nn.LeakyReLU())
        self.linear_relu_cat_prop = nn.Sequential(nn.Linear(11, c_out), nn.LeakyReLU())
        self.linear_relu_struc = nn.Sequential(nn.Linear(128, s_out), nn.LeakyReLU())

        # لایه‌های ترکیب (Projection) برای Cross-Attention
        # Query از ساختار (22 -> 160)
        self.proj_s = nn.Linear(s_out, hidden_dim)
        # Key از متن (22+42=64 -> 160)
        self.proj_dt = nn.Linear(d_out + t_out, hidden_dim)

        if use_anti_mimicry:
            # لایه ویژگی ضدتقلید
            self.linear_relu_antimim = nn.Sequential(nn.Linear(7, am_out), nn.LeakyReLU())
            # Value از عددی+دسته‌ای+ضدتقلید (22+52+22=96 -> 160)
            self.proj_nc = nn.Linear(n_out + c_out + am_out, hidden_dim)
        else:
            # Value از عددی+دسته‌ای (22+52=74 -> 160)
            self.proj_nc = nn.Linear(n_out + c_out, hidden_dim)

        self.attention = MultiHeadAttention(hidden_dim, num_heads, dropout)

        # لایه‌های گرافی (2 لایه مطابق مقاله)
        if gnn_type == "RGCN":
            self.conv1 = RGCNConv(hidden_dim, hidden_dim, num_relations=2)
            self.conv2 = RGCNConv(hidden_dim, hidden_dim, num_relations=2)
        else:
            self.conv1 = SAGEConv(hidden_dim, hidden_dim)
            self.conv2 = SAGEConv(hidden_dim, hidden_dim)

        self.linear_relu_output1 = nn.Sequential(nn.Linear(hidden_dim, hidden_dim), nn.LeakyReLU())
        self.linear_output2 = nn.Linear(hidden_dim, 2)

    def forward(self, des, tweet, num_prop, cat_prop, struc, edge_index, edge_type, antimim=None):
        d = self.linear_relu_des(des)
        t = self.linear_relu_tweet(tweet)
        n = self.linear_relu_num_prop(num_prop)
        c = self.linear_relu_cat_prop(cat_prop)
        s = self.linear_relu_struc(struc)

        # ترکیب Cross-Attention مطابق مقاله پایه
        q = self.proj_s(s)
        k = self.proj_dt(torch.cat((d, t), dim=1))

        # اگر آنتیمیمی روشن باشد، به Value اضافه میشود
        if self.use_anti_mimicry and antimim is not None:
            am = self.linear_relu_antimim(antimim)
            v = self.proj_nc(torch.cat((n, c, am), dim=1))
        else:
            v = self.proj_nc(torch.cat((n, c), dim=1))

        # خروجی Cross-Attention
        x = self.attention(q, k, v).squeeze(1) if self.attention(q, k, v).dim() == 3 else self.attention(q, k, v)

        # 2 لایه گرافی
        if self.gnn_type == "RGCN":
            x = F.relu(self.conv1(x, edge_index, edge_type))
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = self.conv2(x, edge_index, edge_type)
        else:
            x = F.relu(self.conv1(x, edge_index))
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = self.conv2(x, edge_index)

        x = self.linear_relu_output1(x)
        x = self.linear_output2(x)
        return x

# 7 - آموزش، ارزیابی و مقایسه دو مدل

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import pandas as pd

EPOCHS = 100
HIDDEN_DIM = 128
LR = 1e-3
WEIGHT_DECAY = 5e-3

total_nodes = len(train_data) + len(dev_data) + len(test_data)
semantic_all = torch.cat([semantic_train, semantic_dev, semantic_test]).to(device)
num_all = torch.cat([num_train, num_dev, num_test]).to(device)
cat_all = torch.cat([cat_train, cat_dev, cat_test]).to(device)
comm_all = torch.cat([comm_train, comm_dev, comm_test]).to(device)
antimim_all = torch.cat([antimim_train, antimim_dev, antimim_test]).to(device)
y_all = torch.cat([y_train, y_dev, y_test]).to(device)

train_mask = torch.zeros(total_nodes, dtype=torch.bool).to(device)
test_mask = torch.zeros(total_nodes, dtype=torch.bool).to(device)
train_mask[:len(train_data)] = True
test_mask[len(train_data)+len(dev_data):] = True

def init_weights(m):
    if type(m) == nn.Linear:
        nn.init.kaiming_uniform_(m.weight)

def train_and_evaluate(model_name, model):
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    criterion = nn.CrossEntropyLoss()
    model.apply(init_weights)

    print(f"\n--- Training {model_name} ---")
    best_metrics = {'accuracy': 0.0, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0}

    for epoch in range(EPOCHS):
        model.train()
        optimizer.zero_grad()

        out = model(semantic_all[:, :768], semantic_all[:, 768:], num_all, cat_all, comm_all, edge_index, edge_type, antimim_all)
        loss = criterion(out[train_mask], y_all[train_mask])
        loss.backward()
        optimizer.step()

        if (epoch + 1) % 10 == 0:
            model.eval()
            with torch.no_grad():
                logits = model(semantic_all[:, :768], semantic_all[:, 768:], num_all, cat_all, comm_all, edge_index, edge_type, antimim_all)
                preds = logits[test_mask].argmax(dim=1).cpu().numpy()
                true = y_all[test_mask].cpu().numpy()

                acc = accuracy_score(true, preds)
                prec = precision_score(true, preds, zero_division=0)
                rec = recall_score(true, preds, zero_division=0)
                f1 = f1_score(true, preds, zero_division=0)

                if f1 > best_metrics['f1']:
                    best_metrics = {'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1}
                print(f"Epoch {epoch+1:03d} | Loss: {loss.item():.4f} | Acc: {acc:.4f} | F1: {f1:.4f}")

    print(f">>> Best Metrics for {model_name} -> Acc: {best_metrics['accuracy']:.4f}, Prec: {best_metrics['precision']:.4f}, Rec: {best_metrics['recall']:.4f}, F1: {best_metrics['f1']:.4f}")
    return best_metrics

# 1. روش پایه: RGCN + بدون Anti-Mimicry (دقیقاً مطابق BotCF مقاله)
base_model = BotDetector(hidden_dim=HIDDEN_DIM, use_anti_mimicry=False, gnn_type="RGCN").to(device)
base_metrics = train_and_evaluate("Base Model (BotCF - RGCN)", base_model)

# 2. روش پیشنهادی: GraphSAGE + با Anti-Mimicry
proposed_model = BotDetector(hidden_dim=HIDDEN_DIM, use_anti_mimicry=True, gnn_type="GraphSAGE").to(device)
proposed_metrics = train_and_evaluate("Proposed Model (GraphSAGE + Anti-Mimicry)", proposed_model)

# ==========================================
# جدول مقایسه نهایی
# ==========================================
print("\n" + "="*60)
print(" " * 20 + "FINAL COMPARISON TABLE")
print("="*60)

results_df = pd.DataFrame({
    'Model': ['Base Model (BotCF)', 'Proposed Model'],
    'Accuracy': [base_metrics['accuracy'], proposed_metrics['accuracy']],
    'Precision': [base_metrics['precision'], proposed_metrics['precision']],
    'Recall': [base_metrics['recall'], proposed_metrics['recall']],
    'F1-Score': [base_metrics['f1'], proposed_metrics['f1']]
})

results_df_display = results_df.copy()
for col in ['Accuracy', 'Precision', 'Recall', 'F1-Score']:
    results_df_display[col] = results_df_display[col].apply(lambda x: f"{x:.4f}")

print(results_df_display.to_string(index=False))
print("="*60)


--- Training Base Model (BotCF - RGCN) ---
Epoch 010 | Loss: 0.9283 | Acc: 0.5435 | F1: 0.3662
Epoch 020 | Loss: 0.5768 | Acc: 0.6729 | F1: 0.6596
Epoch 030 | Loss: 0.5074 | Acc: 0.7988 | F1: 0.8361
Epoch 040 | Loss: 0.4707 | Acc: 0.8039 | F1: 0.8400
Epoch 050 | Loss: 0.4572 | Acc: 0.8166 | F1: 0.8543
Epoch 060 | Loss: 0.4451 | Acc: 0.8157 | F1: 0.8539
Epoch 070 | Loss: 0.4347 | Acc: 0.8140 | F1: 0.8522
Epoch 080 | Loss: 0.4286 | Acc: 0.8157 | F1: 0.8531
Epoch 090 | Loss: 0.4246 | Acc: 0.8157 | F1: 0.8533
Epoch 100 | Loss: 0.4320 | Acc: 0.8157 | F1: 0.8535
Epoch 110 | Loss: 0.4212 | Acc: 0.8166 | F1: 0.8545
Epoch 120 | Loss: 0.4190 | Acc: 0.8166 | F1: 0.8543
Epoch 130 | Loss: 0.4159 | Acc: 0.8157 | F1: 0.8537
Epoch 140 | Loss: 0.4133 | Acc: 0.8174 | F1: 0.8548
Epoch 150 | Loss: 0.4146 | Acc: 0.8157 | F1: 0.8537
>>> Best Metrics for Base Model (BotCF - RGCN) -> Acc: 0.8174, Prec: 0.7500, Rec: 0.9938, F1: 0.8548

--- Training Proposed Model (GraphSAGE + Anti-Mimicry) ---
Epoch 010 | Los